💡 **Environment:** `clamp-analyses` (R kernel)

# Figure 3: training scale and pathway and genetic-trait recovery

Exported at **180 × 150 mm** with all text at least **5 pt** (Nature Methods: at most 180 × 170 mm, 5–7 pt).

- **a, b**: CLAMPbase pathway (ten replicates each) and trait (ten replicates each) recovery across ARCHS4
  training fractions (1–100%) with a side panel for GTEx. recount2 is not shown.
- **c**: pathway recovery against the number of modules *K*; **d**: the *K* = 1,728 subset of **c**.
- **a, b, d**: ARCHS4 100% vs 75%, 50%, 25% and GTEx (a, b) or vs 75%, 50%, 25% at *K* = 1,728 (d): two-sided, unpaired,
  pooled-variance Student's *t*-tests, BH FDR within each panel.

All plotted source tables are exported to `output/99_panels/fig3/source_data`.

In [ ]:
source(here::here("scripts/panels/main_figure.R"))
suppressPackageStartupMessages({ library(ggpubr) })
set.seed(1)
dirs <- main_figure_dirs("fig3")
source_dir <- dirs$source
FIG_W <- 180
FIG_H <- 150
fraction_levels <- function(x) sprintf("%s%%", sort(unique(x)))

## Panels a and b: recovery across ARCHS4 training fractions and the complete compendia

Panel a uses ten independent CLAMPbase replicates per group (GTEx seeds 123, 456, 789 and 1001-1007);
panel b uses ten (GTEx: the published model plus replicates 2-10).

In [ ]:
coverage_long <- fread(file.path(DATA_DIR, "00_coverage", "coverage_long.csv"))[database != "reactome"]
FDR <- unique(coverage_long$fdr)
stopifnot(length(FDR) == 1L)

ref_tests <- function(d, fractions, ref, alternative = "two.sided") {
  rbindlist(lapply(fractions, function(fv) {
    y <- d[fraction == ref][order(seed)]$recovered
    x <- d[fraction == fv][order(seed)]$recovered
    if (length(x) < 2L || length(y) < 2L) return(NULL)
    # Keep the comparison bracket, but withhold P until reference seeds are independent.
    p_value <- if (uniqueN(y) < 2L) NA_real_ else
      t.test(y, x, paired = FALSE, alternative = alternative, var.equal = TRUE)$p.value
    data.table(group1 = sprintf("%s%%", fv), group2 = sprintf("%s%%", ref), n = length(x), n_ref = length(y),
               p = p_value,
               difference = as.integer(round(mean(y) - mean(x))))
  }))
}

# ARCHS4 fractions and GTEx in one plot so a bracket can join ARCHS4 100% and GTEx.
# Complete-compendium results with three rows are drawn as boxes, including provisional trait copies.
recovery_panel <- function(archs4, gtex, metric, tag, alternative = "two.sided", adjust = TRUE) {
  archs4 <- copy(archs4); gtex <- copy(gtex)
  with100 <- archs4[fraction == 100, .N >= 3L]
  single100 <- if (with100) NULL else archs4[fraction == 100][order(seed)][1]
  if (!with100) archs4 <- archs4[fraction < 100]
  fr <- sort(unique(archs4$fraction))
  archs4[, x := match(fraction, fr)]
  gtex_x <- length(fr) + 1.7                     # GTEx sits after a gap
  single_x <- length(fr) + 1                     # ARCHS4 100% point when not a box
  gtex[, x := gtex_x]
  ref <- max(fr)
  cands <- if (ref == 100) c(75L, 50L, 25L) else c(50L, 25L)
  tests <- ref_tests(archs4, intersect(cands, fr), ref, alternative)
  if (nrow(tests) == 0L) tests <- data.table(group1 = character(), group2 = character(),
                                               n = integer(), n_ref = integer(), p = numeric(), difference = integer())
  gtex_boxes <- gtex[, .N >= 3L]
  if (with100 && gtex_boxes) {
    y <- archs4[fraction == 100]$recovered; x <- gtex$recovered
    # Duplicated trait seeds give a descriptive difference but no valid P value.
    p_value <- if (uniqueN(y) < 2L || uniqueN(x) < 2L) NA_real_ else
      t.test(y, x, alternative = alternative, var.equal = TRUE)$p.value
    tests <- rbind(tests, data.table(group1 = "GTEx", group2 = "100%", n = length(x), n_ref = length(y),
                                     p = p_value,
                                     difference = as.integer(round(mean(y) - mean(x)))))
  }
  all_vals <- c(archs4$recovered, gtex$recovered, single100$recovered)
  span <- diff(range(all_vals))
  xpos <- function(g) ifelse(g == "GTEx", gtex_x, match(as.integer(sub("%", "", g)), fr))
  tests[, `:=`(x1 = xpos(group2), x2 = xpos(group1))]
  tests <- tests[order(abs(x2 - x1))]            # shortest bracket lowest
  top <- max(archs4$recovered, gtex$recovered)
  tests[, p_adj := if (adjust) p.adjust(p, method = "BH") else p]
  tests[, `:=`(y = top + span * (0.06 + 0.17 * seq_len(.N)), label = sprintf("%s = %s, %+d", if (adjust) "q" else "p", ifelse(is.na(p_adj), "pending", nm_p(p_adj)), difference))]
  ylim <- c(min(all_vals) - 0.34 * span, max(tests$y, top) + span * 0.14)

  lab <- function(d) d[, .(recovered = as.integer(round(median(recovered))), eligible = as.integer(round(median(eligible))), n = .N), by = x][
    , label := sprintf("%s\n(%.0f%%)%s", format(recovered, big.mark = ",", trim = TRUE), 100 * recovered / eligible, ifelse(n < 10L & adjust, sprintf("\nn = %d", n), ""))]
  labs_d <- rbind(lab(archs4), lab(gtex), if (!is.null(single100)) lab(single100[, x := single_x]))
  # Boxes coloured by training fraction (config ARCHS4_FRACTION_COLORS), as in panels c and d;
  # GTEx in its compendium colour (config COMPENDIUM_COLORS).
  archs4[, col := FRACTION_COLORS[sprintf("%s%%", fraction)]]
  gtex[, col := COMPENDIUM_COLORS[["GTEx"]]]
  p <- ggplot() +
    geom_boxplot(data = archs4, aes(x, recovered, group = x, fill = col, colour = col),
                 width = 0.6, outlier.shape = NA, alpha = 0.35, linewidth = 0.25) +
    geom_jitter(data = archs4, aes(x, recovered, colour = col), width = 0.10, height = 0, size = 0.5, alpha = 0.8)
  if (gtex_boxes) {
    p <- p + geom_boxplot(data = gtex, aes(x, recovered, group = x, fill = col, colour = col),
                          width = 0.6, outlier.shape = NA, alpha = 0.35, linewidth = 0.25) +
      geom_jitter(data = gtex, aes(x, recovered, colour = col), width = 0.10, height = 0, size = 0.5, alpha = 0.8)
  } else {
    p <- p + geom_point(data = gtex, aes(x, recovered, colour = col), size = 1.4)
  }
  if (!is.null(single100)) {
    p <- p + geom_point(data = single100, aes(x, recovered), colour = FRACTION_COLORS[["100%"]], size = 1.4)
  }
  p <- p + scale_fill_identity() + scale_colour_identity()
  brk <- 0.012 * span
  p <- p +
    geom_text(data = labs_d, aes(x, recovered - 0.17 * span, label = label), size = 5 / .pt, vjust = 1, lineheight = .95) +
    geom_vline(xintercept = (max(archs4$x, if (!is.null(single100)) single_x) + gtex_x) / 2,
               colour = "grey70", linewidth = 0.25, linetype = "dotted") +
    geom_segment(data = tests, aes(x = x1, xend = x2, y = y, yend = y), linewidth = 0.25) +
    geom_segment(data = tests, aes(x = x1, xend = x1, y = y, yend = y - brk), linewidth = 0.25) +
    geom_segment(data = tests, aes(x = x2, xend = x2, y = y, yend = y - brk), linewidth = 0.25) +
    geom_text(data = tests, aes((x1 + x2) / 2, y + brk, label = label), size = 5 / .pt, vjust = 0) +
    scale_x_continuous(breaks = c(seq_along(fr), if (!is.null(single100)) single_x, gtex_x),
                       labels = c(sprintf("%s%%", fr), if (!is.null(single100)) "100%", "GTEx")) +
    coord_cartesian(xlim = c(0.4, gtex_x + 0.6), ylim = ylim, expand = FALSE, clip = "off") +
    labs(x = "Training studies used (ARCHS4)", y = paste("Recovered", metric)) +
    theme_nm()
  fwrite(archs4, file.path(source_dir, paste0(tag, "_archs4_fractions.csv")))
  fwrite(gtex, file.path(source_dir, paste0(tag, "_gtex_replicates.csv")))
  fwrite(tests, file.path(source_dir, paste0(tag, "_tests.csv")))
  p
}

GTEX_SEEDS <- unlist(yaml::read_yaml(here::here("workflow/config/archs4.yaml"))$archs4$coverage$pathway$gtex_seeds)
arch_pathways <- coverage_long[dataset == "archs4" & model == "CLAMPbase", .(
  recovered = sum(recovered_pathways), eligible = sum(eligible_pathways), n_db = .N), by = .(fraction, seed)][n_db == 2L]
gtex_pathways <- coverage_long[dataset == "gtex" & model == "CLAMPbase" & seed > 0L, .(
  recovered = sum(recovered_pathways), eligible = sum(eligible_pathways), n_db = .N), by = seed][n_db == 2L]
gtex_pathways[, rng_seed := GTEX_SEEDS[seed]]
plot_A <- recovery_panel(arch_pathways, gtex_pathways, "pathways", "a", alternative = "two.sided", adjust = TRUE)
options(repr.plot.width = 3.5, repr.plot.height = 2.4)
print(plot_A)

# Traits: ARCHS4 from the coverage trait summaries; GTEx CLAMPbase seed 0 and replicates 2-10.
arch_traits <- fread(file.path(DATA_DIR, "00_coverage/coverage_trait_recovery.csv"))[model == "CLAMPbase"]
stopifnot(nrow(arch_traits) > 0, all(arch_traits$fdr == FDR))
arch_traits <- arch_traits[, .(fraction, seed, recovered = recovered_traits, eligible = eligible_traits)]
gtex_traits <- fread(file.path(DATA_DIR, "00_coverage/final_trait_recovery.csv"))[
  model == "CLAMPbase" & dataset == "gtex", .(seed, recovered = recovered_traits, eligible = eligible_traits)]
stopifnot(nrow(gtex_traits) == 10L, uniqueN(arch_traits[fraction == 100]$seed) == 10L)
gtex_traits[, rng_seed := c(0L, 456L, 789L, 1001:1007)[match(seed, c(0L, 2:10))]]
plot_B <- recovery_panel(arch_traits, gtex_traits, "traits", "b")
print(plot_B)

## Panel c: pathway recovery against the number of modules K

In [ ]:
saturation_long <- fread(file.path(DATA_DIR, "01_saturation", "saturation_long.csv"))[database != "reactome"]
sat <- saturation_long[dataset == "archs4" & model == "CLAMPbase", .(
  recovered = sum(recovered_pathways), eligible = sum(eligible_pathways), n_db = .N), by = .(fraction, k, seed)][n_db == 2L]
sat[, k_label := factor(k, levels = sort(unique(k)))]
sat[, fraction_label := factor(sprintf("%d%%", fraction), levels = fraction_levels(fraction))]
sat_k <- droplevels(sat[fraction < 100])

DODGE <- position_dodge(width = 0.85)
dodge_offset <- function(g) (as.integer(g) - (nlevels(g) + 1) / 2) * (0.85 / nlevels(g))
means_C <- sat_k[, .(recovered = mean(recovered)), by = .(fraction_label, k_label)]
means_C[, x := as.integer(k_label) + dodge_offset(fraction_label)]

plot_C <- ggplot(sat_k, aes(k_label, recovered, fill = fraction_label, colour = fraction_label)) +
  geom_boxplot(position = DODGE, width = 0.75, outlier.shape = NA, alpha = 0.35, linewidth = 0.25) +
  geom_point(position = position_jitterdodge(jitter.width = 0.12, dodge.width = 0.85),
             size = 0.5, alpha = 0.65, show.legend = FALSE) +
  geom_line(data = means_C, aes(x = x, group = fraction_label), linetype = "dashed", linewidth = 0.3,
            show.legend = FALSE) +
  scale_fill_manual(values = FRACTION_COLORS) +
  scale_colour_manual(values = FRACTION_COLORS) +
  scale_y_continuous(expand = expansion(mult = c(0.04, 0.08))) +
  labs(x = "Gene modules (K)", y = "Recovered pathways", fill = "Training studies used",
       colour = "Training studies used") +
  theme_nm() +
  theme(legend.position = "bottom", legend.key.size = unit(2.5, "mm")) +
  guides(fill = guide_legend(nrow = 1), colour = guide_legend(nrow = 1))
legend_C <- get_legend(plot_C)
plot_C <- plot_C + theme(legend.position = "none")
fwrite(sat_k, file.path(source_dir, "c_pathways_by_k.csv"))
print(plot_C)

## Panel d: the K = 1,728 subset of panel c, with the full compendium

1% subsamples have an SVD rank below 1,728, so their K = 1,728 fits start from an additional SVD with 1,728
components (Methods). The 100% column is the ten full-compendium replicates, whose native rank is K = 1,728;
it is the reference for the 75%, 50% and 25% comparisons.

In [ ]:
sat_lastk <- droplevels(sat[k == max(k)])
tests_D <- ref_tests(sat_lastk, fractions = c(75L, 50L, 25L), ref = 100L, alternative = "two.sided")
tests_D[, p_adj := p.adjust(p, method = "BH")]
lv_D <- levels(sat_lastk$fraction_label)
span_D <- diff(range(sat_lastk$recovered))
tests_D <- tests_D[order(-match(group1, lv_D))]
tests_D[, `:=`(x = match(group1, lv_D), xend = match(group2, lv_D),
               y = max(sat_lastk$recovered) + span_D * (0.08 + 0.22 * seq_len(.N)),
               label = sprintf("q = %s, %+d", nm_p(p_adj), difference))]
plot_D <- ggplot(sat_lastk, aes(fraction_label, recovered, fill = fraction_label, colour = fraction_label)) +
  geom_boxplot(width = 0.6, outlier.shape = NA, alpha = 0.35, linewidth = 0.25) +
  geom_jitter(width = 0.12, size = 0.5, alpha = 0.65) +
  geom_segment(data = tests_D, aes(x = x, xend = xend, y = y, yend = y), inherit.aes = FALSE,
               colour = "#444444", linewidth = 0.25) +
  geom_segment(data = tests_D, aes(x = x, xend = x, y = y, yend = y - span_D * 0.02), inherit.aes = FALSE,
               colour = "#444444", linewidth = 0.25) +
  geom_segment(data = tests_D, aes(x = xend, xend = xend, y = y, yend = y - span_D * 0.02), inherit.aes = FALSE,
               colour = "#444444", linewidth = 0.25) +
  geom_text(data = tests_D, aes(x = (x + xend) / 2, y = y + span_D * 0.03, label = label), inherit.aes = FALSE,
            size = 5 / .pt, vjust = 0) +
  scale_fill_manual(values = FRACTION_COLORS, guide = "none") +
  scale_colour_manual(values = FRACTION_COLORS, guide = "none") +
  scale_y_continuous(expand = expansion(mult = c(0.04, 0.10))) +
  labs(x = "Training studies used (K = 1,728)", y = NULL) +
  theme_nm()
fwrite(sat_lastk, file.path(source_dir, "d_pathways_k1728.csv"))
fwrite(tests_D, file.path(source_dir, "d_reference_tests.csv"))
print(plot_D)

## Assemble and save

In [ ]:
# d zooms into the K = 1,728 group of c: dashed box in c, connectors to the corners of d's panel.
panel_box <- function(p, w_mm, h_mm) {
  grDevices::pdf(NULL, width = w_mm / 25.4, height = h_mm / 25.4)
  g <- ggplotGrob(p)
  w <- grid::convertWidth(g$widths, "mm", valueOnly = TRUE)
  h <- grid::convertHeight(g$heights, "mm", valueOnly = TRUE)
  grDevices::dev.off()
  pan <- g$layout[g$layout$name == "panel", ]
  axl <- g$layout[g$layout$name == "axis-l", ]
  pw <- w_mm - sum(w); ph <- h_mm - sum(h)
  pp <- ggplot_build(p)$layout$panel_params[[1]]
  list(x0 = sum(w[seq_len(pan$l - 1)]), w = pw, axis_x0 = sum(w[seq_len(axl$l - 1)]),
       y0 = h_mm - sum(h[seq_len(pan$t - 1)]) - ph, h = ph, xr = pp$x.range, yr = pp$y.range)
}
ROW_H <- c(1, 1, 0.07); REL_W <- c(1.35, 1); TAG_X <- 0.015; TAG_TOP <- 0.94
k_last <- sat_k[k == max(k), range(recovered)]
k_pad <- diff(k_last) * 0.06
zoom <- list(x = nlevels(sat_k$k_label) + c(-0.47, 0.47), y = k_last + c(-k_pad, k_pad))
plot_C <- plot_C + annotate("rect", xmin = zoom$x[1], xmax = zoom$x[2], ymin = zoom$y[1], ymax = zoom$y[2],
                            fill = NA, colour = "grey35", linewidth = 0.25, linetype = "dashed")
row_y0 <- FIG_H * ROW_H[3] / sum(ROW_H); row_h <- FIG_H * ROW_H[2] / sum(ROW_H)
slot_w <- FIG_W * REL_W / sum(REL_W); slot_x <- c(0, slot_w[1])
plot_w <- slot_w * (1 - TAG_X); plot_x <- slot_x + slot_w * TAG_X; plot_h <- row_h * TAG_TOP
box_C <- panel_box(plot_C, plot_w[1], plot_h); box_D <- panel_box(plot_D, plot_w[2], plot_h)
to_y <- function(b, y) row_y0 + b$y0 + (y - b$yr[1]) / diff(b$yr) * b$h
zoom_x0 <- plot_x[1] + box_C$x0 + (zoom$x[2] - box_C$xr[1]) / diff(box_C$xr) * box_C$w
zoom_x1 <- plot_x[2] + box_D$axis_x0 - 0.8  # stop left of the tick labels

row1 <- plot_grid(add_tag(plot_A, "a"), add_tag(plot_B, "b"), nrow = 1)
row2 <- plot_grid(add_tag(plot_C, "c"), add_tag(plot_D, "d"), nrow = 1, rel_widths = REL_W)
fig3 <- plot_grid(row1, row2, legend_C, ncol = 1, rel_heights = ROW_H)
for (i in 1:2)
  fig3 <- fig3 + draw_line(x = c(zoom_x0, zoom_x1) / FIG_W,
                           y = c(to_y(box_C, zoom$y[i]), row_y0 + box_D$y0 + box_D$h * (i - 1)) / FIG_H,
                           colour = "grey35", linewidth = 0.25, linetype = "dashed")
writeLines(c(sprintf("Figure 3: %d x %d mm; minimum text 5 pt.", FIG_W, FIG_H),
             "a, b: CLAMPbase independent pathway (a, ten per group; GTEx seeds 123, 456, 789, 1001-1007) and trait (b, ten; GTEx published model plus replicates 2-10) replicates.",
             "c, d: CLAMPbase fixed-rank refits (ten per cell); d is the K = 1,728 subset of c plus the ten 100% replicates (native K = 1,728).",
             "Comparisons: a and b ARCHS4 100% vs 75%, 50%, 25% and GTEx; d 100% vs 75%, 50%, 25%. a, b and d: two-sided, unpaired, pooled-variance Student t-tests, BH FDR within panel."),
           file.path(source_dir, "README.txt"))
save_main_figure(fig3, "fig3", FIG_W, FIG_H)
options(repr.plot.width = FIG_W / 25.4, repr.plot.height = FIG_H / 25.4)
print(fig3)